[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WesamKhallaf73/pytorch-ground-up-notebooks/blob/main/notebooks/L12.ipynb)

# Lecture 12 — The Layer Catalog & Composing Blocks

*The handful of layers you’ll use every day — and how to snap them into blocks you write once and reuse everywhere.*

**PyTorch from Ground Up · Volume I · Chapters 20–21** — companion notebook.

**How to use it:** run the cells from top to bottom. Before each code cell, *predict* what it will print — then run it and compare with the expected output (click ▸ to open it). Every exercise from the lecture is at the end.

By the end you will be able to:
- Use `nn.Linear` and count its parameters with `out × in + out`.
- Explain why networks need activations, and pick one.
- Build straight-line models with `nn.Sequential` — and know where it stops.
- Place Dropout and normalization correctly, and tell BatchNorm from LayerNorm.
- Package a repeated pattern into a reusable block.
- Write a residual block, and store sub-modules in `ModuleList` / `ModuleDict`.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
print('PyTorch', torch.__version__)

## 01 · Linear: the universal building block

You have the container. Now: what goes inside? A **handful of layers covers nearly everything**. `Linear`, the main activations, `Sequential`, `Dropout` and two normalisation layers.

<sub>Source: lecture slides 3–6</sub>

**`linear.py`** — predict the output, then run.

In [ ]:
layer = nn.Linear(8, 4)
x = torch.randn(2, 8)          # batch of 2, 8 features
out = layer(x)
print(out.shape)               # (2, 4)

<details><summary>▸ Expected output</summary>

```
torch.Size([2, 4])
```
</details>

## 02 · Activations

Two linear layers in a row are **still just one linear map**. Activations add the bend. Without something in between, a stack of Linear layers is no more expressive than one. The nonlinearity is what lets a network learn curved patterns.

<sub>Source: lecture slides 7–10</sub>

🧪 *Interactive lab in the app:* **Six activations** — try it there to see this idea move.

**`activations.py`** — predict the output, then run.

In [ ]:
x = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])
print(nn.ReLU()(x))            # [0, 0, 0, 1, 2]
print(nn.LeakyReLU(0.1)(x))    # [-0.2, -0.1, 0, 1, 2]
print(nn.Sigmoid()(x))         # [0.12, 0.27, 0.50, 0.73, 0.88]

<details><summary>▸ Expected output</summary>

```
tensor([0., 0., 0., 1., 2.])
tensor([-0.2000, -0.1000,  0.0000,  1.0000,  2.0000])
tensor([0.1192, 0.2689, 0.5000, 0.7311, 0.8808])
```
</details>

> ⚠️ **Watch out — Activations have no learnable parameters**  
> Activations are pure functions — nothing inside them learns.
>
> You can call them as module objects (`nn.ReLU()`) or as functions from `torch.nn.functional` (`F.relu(x)`) — both work. Use the module form in `__init__` when building with `Sequential`; use the functional form in `forward` when you want inline calls.

## 03 · Sequential: layers in a line

A straight line of layers with no branches? `nn.Sequential` — **no class, no `forward`**.

<sub>Source: lecture slides 11–14</sub>

**`sequential.py`** — predict the output, then run.

In [ ]:
model = nn.Sequential(
    nn.Linear(1, 32),
    nn.ReLU(),
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Linear(16, 1),
)
print(model)
x = torch.randn(4, 1)
print(model(x).shape)          # (4, 1)

<details><summary>▸ Expected output</summary>

```
Sequential(
  (0): Linear(in_features=1, out_features=32, bias=True)
  (1): ReLU()
  (2): Linear(in_features=32, out_features=16, bias=True)
  (3): ReLU()
  (4): Linear(in_features=16, out_features=1, bias=True)
)
torch.Size([4, 1])
```
</details>

🧪 *Interactive lab in the app:* **Count a Sequential stack** — try it there to see this idea move.

## 04 · Dropout: regularization by forgetting

`Dropout(p)` randomly silences a fraction `p` of activations — so no neuron can **carry the whole load**. Knowledge gets spread across many neurons, and the network overfits less.

<sub>Source: lecture slides 15–19</sub>

🧪 *Interactive lab in the app:* **Dropout — only during training** — try it there to see this idea move.

**`placement.py`** — predict the output, then run.

In [ ]:
# Typical placement in a model
model = nn.Sequential(
    nn.Linear(64, 128),
    nn.ReLU(),
    nn.Dropout(0.3),                # after activation
    nn.Linear(128, 10),
)

> ⚠️ **Watch out — After the activation, never on the output**  
> Dropout goes **after** the activation, not before.
>
> And never use it on the final output layer — you want the model’s full capacity for predictions, not a randomly crippled version.

## 05 · Normalization, and a model with all the pieces

Deep stacks let activations drift and grow. **Normalisation rescales them** at every layer. Roughly zero mean and unit variance — which stabilises and speeds up training.

<sub>Source: lecture slides 20–26</sub>

🧪 *Interactive lab in the app:* **BatchNorm vs LayerNorm** — try it there to see this idea move.

**`classifier.py`** — predict the output, then run.

In [ ]:
class Classifier(nn.Module):
    def __init__(self, in_dim, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, n_classes),
        )

    def forward(self, x):
        return self.net(x)

model = Classifier(784, 10)
print(model)

## 06 · Think in blocks

Now think in **blocks**: small modules you write once and snap together like bricks. A residual block, a transformer layer, an encoder — each is an `nn.Module` holding other modules, and `.parameters()` still finds everything.

<sub>Source: lecture slides 28–32</sub>

**`dense_block.py`** — predict the output, then run.

In [ ]:
class DenseBlock(nn.Module):
    def __init__(self, in_dim, out_dim, drop=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.BatchNorm1d(out_dim),
            nn.ReLU(),
            nn.Dropout(drop),
        )
    def forward(self, x):
        return self.net(x)

class Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.block1 = DenseBlock(784, 256)
        self.block2 = DenseBlock(256, 128)
        self.block3 = DenseBlock(128, 64)
        self.head   = nn.Linear(64, 10)
    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        return self.head(x)

print(Classifier())

<details><summary>▸ Expected output</summary>

```
Classifier(
  (block1): DenseBlock(...)
  (block2): DenseBlock(...)
  (block3): DenseBlock(...)
  (head): Linear(in_features=64, out_features=10)
)
```
</details>

## 07 · The residual connection

One of the most important ideas in deep learning: don’t compute f(x) — compute **x + f(x)**.

<sub>Source: lecture slides 33–37</sub>

**`res_block.py`** — predict the output, then run.

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim),
            nn.ReLU(),
            nn.Linear(dim, dim),
        )
    def forward(self, x):
        return x + self.net(x)        # the skip

## 08 · ModuleList, ModuleDict and unlimited nesting

N identical blocks built in a loop? A plain list hides them. `nn.ModuleList` **registers** them.

<sub>Source: lecture slides 38–46</sub>

**`deep_res.py`** — predict the output, then run.

In [ ]:
class DeepRes(nn.Module):
    def __init__(self, dim, n_blocks):
        super().__init__()
        self.blocks = nn.ModuleList(
            [ResBlock(dim) for _ in range(n_blocks)]
        )
        self.head = nn.Linear(dim, 10)
    def forward(self, x):
        for block in self.blocks:
            x = block(x)
        return self.head(x)

model = DeepRes(64, 5)
print(len(list(model.parameters())))    # all found

<details><summary>▸ Expected output</summary>

```
22
```
</details>

🧪 *Interactive lab in the app:* **Plain list vs nn.ModuleList** — try it there to see this idea move.

> ⚠️ **Watch out — Never a plain list or dict**  
> Never store sub-modules in a plain Python list or dict. Use `nn.ModuleList` or `nn.ModuleDict`.
>
> They register their contents so that `.parameters()`, `.to(device)`, and saving/loading all work correctly. This is one of the most common silent mistakes in custom models.

**`multi_head.py`** — predict the output, then run.

In [ ]:
class MultiHead(nn.Module):
    def __init__(self, in_dim, tasks):
        super().__init__()
        self.backbone = nn.Linear(in_dim, 64)
        self.heads = nn.ModuleDict({
            name: nn.Linear(64, n)
            for name, n in tasks.items()
        })
    def forward(self, x, task):
        x = torch.relu(self.backbone(x))
        return self.heads[task](x)

model = MultiHead(784, {"digit": 10, "odd": 2})

## The rule — worth remembering

**The layer catalog — six rules**

1. `Linear(in, out)`: the universal connector. y = xWᵀ + b.
2. ReLU is the default activation. Use GELU in transformers, LeakyReLU to avoid dead neurons.
3. `Sequential` chains layers without writing a class. Drop back to a custom Module for branching.
4. `Dropout(p)` randomly zeroes p fraction during training. Place it after the activation, never on the final layer.
5. BatchNorm for CNNs and feedforward nets. LayerNorm for transformers and RNNs.
6. A common block: Linear → Norm → Activation → Dropout.

**Composing blocks — six rules**

1. Package repeated patterns into a reusable block — a small `nn.Module`.
2. A residual block computes x + f(x). The skip connection lets gradients flow through deep networks.
3. Use `nn.ModuleList` for a variable number of sub-modules created in a loop.
4. Use `nn.ModuleDict` for named sub-modules you select at runtime.
5. Never use a plain Python list or dict for sub-modules — they won’t be registered.
6. Nesting is unlimited. `.parameters()` recurses through every level.

## Pause and try

Work each exercise in the empty cell below it. Open the worked answer only after you have tried.

### Exercise 01 · A three-layer Sequential

Build a Sequential model with 3 layers: 8 → 32 → 16 → 4, with ReLU between each. Print it and count its parameters.

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

(8·32 + 32) + (32·16 + 16) + (16·4 + 4) = 288 + 528 + 68 = **884** parameters. Check it in the counter above.
</details>

### Exercise 02 · Three residual blocks

Stack three `ResBlock(64)` modules in sequence with a final `nn.Linear(64, 10)`. Pass a (8, 64) batch through the model. Print the output shape and count total parameters.

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

Output shape (8, 10). Each ResBlock holds 2 × (64·64 + 64) = 8,320 parameters; three of them plus the head (64·10 + 10 = 650) give **25,610**.
</details>

### Exercise 03 · Ch 20 · Ex 2

Pass `[-3, -1, 0, 1, 3]` through ReLU, LeakyReLU(0.2), GELU, Sigmoid. Compare outputs.

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

ReLU: [0, 0, 0, 1, 3]. LeakyReLU(0.2): [−0.6, −0.2, 0, 1, 3]. GELU: ≈ [−0.004, −0.159, 0, 0.841, 2.996]. Sigmoid: ≈ [0.047, 0.269, 0.5, 0.731, 0.953]. They agree for positives (except Sigmoid’s squashing) and disagree on the negatives.
</details>

### Exercise 04 · Ch 20 · Ex 4

Dropout p=0.5, same tensor five times — different each time? Switch to eval mode — what changes?

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

In train mode each pass zeroes a different random half and scales the survivors by 2. After `drop.eval()` the output equals the input every time.
</details>

### Exercise 05 · Ch 21 · Ex 2 & 3

Build a model with 4 `ResBlock(32)` via `ModuleList`. Count parameters, confirm all tracked. Then replace `ModuleList` with a plain list. Does `parameters()` still find everything? What’s the count now?

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

With `ModuleList`: 4 × 2 × (32·32 + 32) = 8,448 block parameters, in 16 tensors. With a plain list, `parameters()` finds none of them — only whatever else is registered (e.g. the head). The model still runs, which is what makes the bug silent.
</details>

### Interview-style

(a) Why do networks need activations between linear layers, and what’s the practical difference between BatchNorm and LayerNorm? — four sentences.

_Write your answer here before opening the model answer._

<details><summary>▸ Model answer</summary>

Two linear layers in a row collapse into a single linear map, so without a nonlinearity a deep stack is no more expressive than one layer. Activations add the bend that lets a network model curved patterns. BatchNorm normalizes each feature across the batch, so it depends on batch statistics and behaves differently in train and eval mode. LayerNorm normalizes each sample across its own features, so it is independent of the batch — which is why it suits transformers, RNNs and small batches.
</details>

### Interview-style

(b) What is a residual connection, and why does it help train deep networks? — two sentences. And: why must you use `nn.ModuleList` instead of a Python list for sub-modules? — one sentence.

_Write your answer here before opening the model answer._

<details><summary>▸ Model answer</summary>

A residual connection adds a block’s input to its output, computing x + f(x) instead of f(x). The skip path gives gradients a direct route backward, so very deep networks keep learning even when individual blocks learn slowly.

A plain list doesn’t register its contents, so `.parameters()`, `.to(device)` and saving would all miss those sub-modules and their weights would never train.
</details>